# Detecção de faces falsas com CNN

Experimento de classificação binária com imagens do CIPLab. As imagens reais recebem o rótulo 1 e as manipuladas, 0.

In [ ]:
%pip -q install kagglehub

In [ ]:
from pathlib import Path
import kagglehub

# Preencha com a pasta extraída se já tiver baixado a base.
PASTA_LOCAL = None
raiz = Path(PASTA_LOCAL) if PASTA_LOCAL else Path(
    kagglehub.dataset_download("ciplab/real-and-fake-face-detection")
)
pastas = sorted(
    (p.parent for p in raiz.rglob("training_real")
     if p.is_dir() and (p.parent / "training_fake").is_dir()),
    key=lambda p: (len(p.parts), str(p))
)
if not pastas:
    raise FileNotFoundError("Confira se a base foi extraída e contém training_real e training_fake.")
pasta_base = pastas[0]
print(pasta_base)

## Preparação das imagens

In [ ]:
#Imports
import os
from PIL import Image
import matplotlib.pyplot as plt

#Plotar
def show_samples(dataset_path, n=5):
    files = os.listdir(dataset_path)
    images = [os.path.join(dataset_path, f) for f in files if f.lower().endswith(('jpg', 'png'))][:n]

    plt.figure(figsize=(15,3))
    for i, img_path in enumerate(images):
        img = Image.open(img_path)
        plt.subplot(1, n, i+1)
        plt.imshow(img)
        plt.title(os.path.basename(dataset_path))
        plt.axis('off')
    plt.show()


show_samples(pasta_base / "training_real")
show_samples(pasta_base / "training_fake")

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from sklearn.model_selection import train_test_split

# Parâmetros
IMG_SIZE = (128, 128)

def load_images_from_folder(folder, label):
    data = []
    for filename in os.listdir(folder):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            path = os.path.join(folder, filename)
            img = load_img(path, target_size=IMG_SIZE)
            img = img_to_array(img) / 255.0
            data.append((img, label))
    return data

# Carregar imagens com labels
real_data = load_images_from_folder(pasta_base / "training_real", 1)
fake_data = load_images_from_folder(pasta_base / "training_fake", 0)

# Combinar e embaralhar
all_data = real_data + fake_data
np.random.shuffle(all_data)

# Separar features e labels
X, y = zip(*all_data)
X = np.array(X)
y = np.array(y)

# Treino/validação
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y)

## Treinamento e validação

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

# Treinamento
history = model.fit(X_train, y_train, epochs=10, batch_size=32,
                    validation_data=(X_val, y_val))

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(history.history['accuracy'], label='Treino')
plt.plot(history.history['val_accuracy'], label='Validação')
plt.title('Acurácia')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np

# Prever no conjunto de validação
y_pred_prob = model.predict(X_val)
y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

# Matriz de confusão
cm = confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Falso", "Real"])
disp.plot(cmap='Blues')
plt.title("Matriz de Confusão")
plt.show()

In [ ]:
import random

#Escolher uma imagem aleatória do conjunto de validação
idx = random.randint(0, len(X_val) - 1)
img = X_val[idx]
true_label = y_val[idx]

#Fazer a previsão
prediction = model.predict(np.expand_dims(img, axis=0))[0][0]
predicted_label = 1 if prediction >= 0.5 else 0

#Mostrar imagem e resultado
plt.figure()
plt.imshow(img)
plt.axis('off')
plt.title(f'Verdadeiro: {"Real" if true_label == 1 else "Fake"} | Previsto: {"Real" if predicted_label == 1 else "Fake"}\nScore da classe real: {prediction:.2f}')
plt.show()

## Inspeção visual

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from skimage import feature
import random

#Função para exibir imagens lado a lado
def show_images(images, titles, cmap='gray'):
    plt.figure(figsize=(10, 5))
    for i, (img, title) in enumerate(zip(images, titles)):
        plt.subplot(1, len(images), i + 1)
        plt.imshow(img, cmap=cmap)
        plt.title(title)
        plt.axis('off')
    plt.show()

#Função para aplicar as transformações em imagens do numpy (X_val)
def process_numpy_image(image_array):
    #Se for RGB, converter para GRAY
    if len(image_array.shape) == 3 and image_array.shape[-1] == 3:
        image_gray = cv2.cvtColor((image_array * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    else:
        image_gray = (image_array * 255).astype(np.uint8)

    #Detecção de Bordas com Canny
    edges = cv2.Canny(image_gray, 100, 200)
    edges_normalized = edges.astype(np.float32) / 255.0

    #Textura com LBP
    lbp = feature.local_binary_pattern(image_gray, P=8, R=1, method='uniform')

    #Detecção de Keypoints com SIFT
    sift = cv2.SIFT_create()
    keypoints = sift.detect(image_gray, None)
    image_with_keypoints = cv2.drawKeypoints(
        cv2.cvtColor(image_gray, cv2.COLOR_GRAY2BGR),
        keypoints,
        None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
    )

    return image_gray, edges_normalized, lbp, image_with_keypoints

#Selecionar imagem aleatória do conjunto X_val
idx = random.randint(0, len(X_val) - 1)
img = X_val[idx]
true_label = y_val[idx]

#Processar a imagem
original_gray, edges, lbp, keypoints_image = process_numpy_image(img)

#Exibir a imagem e suas transformações
titles = [
    f'Original {"Real" if true_label == 1 else "Fake"}',
    'Canny Edges',
    'LBP',
    'SIFT Keypoints'
]

show_images([original_gray, edges, lbp, keypoints_image], titles)

In [ ]:
import tensorflow as tf

idx = random.randrange(len(X_val))
image = X_val[idx]
true_label = y_val[idx]
ultima_conv = [camada for camada in model.layers if isinstance(camada, Conv2D)][-1]
ativacoes_modelo = tf.keras.Model(model.inputs, [ultima_conv.output, model.outputs[0]])

with tf.GradientTape() as tape:
    ativacoes, previsoes = ativacoes_modelo(image[None, ...], training=False)
    score_real = previsoes[:, 0]
gradientes = tape.gradient(score_real, ativacoes)
pesos = tf.reduce_mean(gradientes, axis=(1, 2), keepdims=True)
mapa = tf.nn.relu(tf.reduce_sum(ativacoes * pesos, axis=-1))[0]
mapa = tf.math.divide_no_nan(mapa, tf.reduce_max(mapa))
cam = tf.image.resize(mapa[..., None], image.shape[:2])[..., 0].numpy()
pred_prob = float(score_real[0])
pred_label = "Real" if pred_prob >= 0.5 else "Fake"
image_display = image
# Mostrar imagem original e Grad-CAM sobreposta
plt.figure(figsize=(10,4))

plt.subplot(1, 2, 1)
plt.imshow(image_display)
plt.axis('off')
plt.title(f"Original: {'Real' if true_label==1 else 'Fake'}")

plt.subplot(1, 2, 2)
plt.imshow(image_display)
plt.imshow(cam, cmap='jet', alpha=0.5)  # sobreposição do Grad-CAM
plt.axis('off')
plt.title(f"Grad-CAM da classe real\nPredição: {pred_label} ({pred_prob:.2f})")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Garantir que as previsões sejam 1D
y_pred_prob = y_pred_prob.ravel()

# Garantir que os labels verdadeiros sejam 1D
y_true = y_val.ravel()

# Criar DataFrame
df_scores = pd.DataFrame({
    'score': y_pred_prob,
    'classe': ['Real' if y == 1 else 'Falsa' for y in y_true]
})

plt.figure(figsize=(8, 5))
sns.histplot(
    data=df_scores, x='score', hue='classe',
    kde=True, bins=30, palette=['red', 'blue'], alpha=0.6
)
plt.title('Distribuição dos scores de saída da CNN por classe')
plt.xlabel('Score da predição (0 = Falsa, 1 = Real)')
plt.ylabel('Frequência')
plt.xlim(0, 1)
plt.grid(True)
plt.tight_layout()
plt.show()